# Generate Posterior Predictions (Food)

Author:      David Izydorczyk 

In [1]:
import numpy as np
import numpy.random as rng


import matplotlib.pyplot as plt
import seaborn as sns


import os
if "KERAS_BACKEND" not in os.environ:
    # set this to "torch", "tensorflow", or "jax"
    os.environ["KERAS_BACKEND"] = "jax"

import keras
import bayesflow as bf
import pandas as pd



from plotnine import (
    theme_set,
    theme_bw
)
theme_set(theme_bw())



import utils.helper_functions as fn
import utils.model_functions as mf

# makes sure mf is loaded correctly
import importlib
importlib.reload(mf) 
importlib.reload(fn) 

# avoid scientific notation for outputs
np.set_printoptions(suppress=True)

INFO:bayesflow:Using backend 'jax'


# 1 Load Design Data 

In [2]:
df = pd.read_csv('..\..\Materials\design_data_food.csv', sep=';',decimal=",")

all_cues = df[[f'V{i}' for i in range(1, 15)]].to_numpy(dtype=float)
all_crit = df[['crit']].to_numpy(dtype=float).flatten()

# Get Exemplar Set 
exemplars = df.loc[df['training'] == 1,:] 
ex_cues   = exemplars[[f'V{i}' for i in range(1, 15)]].to_numpy(dtype=float)
ex_crit   = exemplars[['crit']].to_numpy(dtype=float).flatten()
ex_IDs    = exemplars[["ID"]].to_numpy(dtype=float).squeeze().astype(int)

# Get Data from only Testing items
testing = df.loc[df['training'] == 0,:]

# Get the IDs
test_IDs = testing[["ID"]].to_numpy(dtype=float).squeeze().astype(int)

# Extract cues of the testing stimuli
cues     = testing[[f'V{i}' for i in range(1, 15)]].to_numpy(dtype=float)

# Make cue dictionary
dict_cues = {f"cue_{i}": cues[:, i] for i in range(cues.shape[1])}

# Geet number of trials and number of dimensions
n_trials, n_dim     = cues.shape

print("n_trials:", n_trials)
print("n_dim:"   , n_dim)

n_trials: 68
n_dim: 14


In [3]:
# Set general parameters
rate = 0.25

In [4]:
# Define positional encodings
position_encodings = np.linspace(0, 1, n_trials, dtype=np.float32)

# 2 Define Models

## 2.1 CAM - Cue Abstraction (Rule) Model 

In [20]:
def prior_CAM(n_dim = n_dim, rate = rate):

    # Init weight parameters
    w = np.zeros(n_dim+1)

    # Intercept
    w[0]   = rng.normal(21.7, 20)

    # Dimension weights (Importance)
    w[1:]  = rng.normal(0, 25, size = n_dim)

    sigma  = rng.exponential(1/rate)

    return dict(w=w, sigma=sigma)

In [21]:
def model_CAM(w, sigma, cues=cues, p=position_encodings):

    n_trials, _   = cues.shape
  
    # Pre-allocate the output matrix
    pred_crit = mf.CAM_experiment(w,cues)

    # Simulate responses 
    x = fn.truncnorm_r(mean=pred_crit, sd=sigma, low=0, upp=100, size=n_trials)
    
    return dict(x=x, p=p)  


In [22]:
simulator_CAM = bf.make_simulator([prior_CAM, model_CAM])

In [23]:
adapter_CAM = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .constrain("sigma", lower=0)
    .as_time_series(["x","p"])
    .standardize()
    .concatenate(["w", "sigma"], into="inference_variables")
    .concatenate(["x", "p"], into="summary_variables")
)


summary_network   = bf.networks.SetTransformer(summary_dim=30)

inference_network = bf.networks.CouplingFlow()


workflow_CAM = bf.BasicWorkflow(
    simulator         = simulator_CAM,
    adapter           = adapter_CAM,
    summary_network   = summary_network,
    inference_network = inference_network,
)


## 2.2 GCM - Exemplar Model


In [6]:
def prior_GCM(n_dim = n_dim, rate = rate):

    c      = rng.exponential(1/0.1)
    w      = rng.dirichlet(np.ones(n_dim), size = 1)*n_dim  
    sigma  = rng.exponential(1/rate)

    return dict(c=c, w=w.squeeze(), sigma=sigma)

In [7]:
def model_GCM(c, w, sigma, cues=cues, ex_cues=ex_cues, ex_crit=ex_crit, p=position_encodings):
  
    n_trials = cues.shape[0]

    # Make predictions based on the GCM model for all trials at once
    pred_crit = mf.GCM_experiment(cues, ex_cues, ex_crit, w, c)

    # Simulate responses
    x = fn.truncnorm_r(mean=pred_crit, sd=sigma, low=0, upp=100, size=n_trials)

    return dict(x=x,p=p) 

In [8]:
simulator_GCM = bf.make_simulator([prior_GCM, model_GCM])

In [9]:
adapter_GCM = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .as_time_series(["x","p"])
    .constrain("sigma", lower=0)
    .constrain("c", lower = 0)
    .constrain("w", lower = 0, upper = n_dim)
        
    .standardize()

    .concatenate(["c", "w", "sigma"], into="inference_variables")
    .concatenate(["x", "p"], into="summary_variables")

   
)

summary_network   = bf.networks.SetTransformer(summary_dim=35)

inference_network = bf.networks.CouplingFlow(subnet="mlp", transform="spline")


workflow_GCM = bf.BasicWorkflow(
    simulator         = simulator_GCM,
    adapter           = adapter_GCM,
    summary_network   = summary_network,
    inference_network = inference_network,
)


## 2.3 RulExJ Model


In [10]:
def prior_RULEXJ(n_dim = n_dim, rate = rate):

    # Rule part
    CAM_pars = prior_CAM(n_dim = n_dim, rate = rate)

    # Exemplar part
    GCM_pars = prior_GCM(n_dim = n_dim, rate = rate)

    # Blending
    a        = rng.uniform(0,1)

    return dict(alpha = a, w_CAM=CAM_pars["w"], c = GCM_pars["c"], w_GCM = GCM_pars["w"], sigma=CAM_pars["sigma"])

In [11]:
def model_RULEXJ(alpha, w_CAM, c, w_GCM, sigma,  cues=cues, ex_cues=ex_cues, ex_crit=ex_crit, p=position_encodings):
    
    n_trials, _   = cues.shape
    
    pred_CAM = mf.CAM_experiment(w_CAM,cues)

    pred_GCM = mf.GCM_experiment(cues, ex_cues, ex_crit,w_GCM,c)

    pred_RULEXJ = alpha*pred_CAM + (1-alpha)*pred_GCM

    # Simulate responses 
    x = fn.truncnorm_r(mean=pred_RULEXJ, sd=sigma, low=0, upp=100, size=n_trials)

    return dict(x=x, p=p)

In [12]:
simulator_RULEXJ = bf.make_simulator([prior_RULEXJ, model_RULEXJ])

In [13]:
adapter_RULEXJ = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .as_time_series(["x","p"])

    .constrain("sigma", lower=0)
    .constrain("c", lower = 0)
    .constrain("w_GCM", lower = 0, upper = n_dim)
    .constrain("alpha", lower = 0, upper = 1)
    
    
    .standardize()

    .concatenate(["alpha","w_CAM","c", "w_GCM", "sigma"], into="inference_variables")
    .concatenate(["x", "p"], into="summary_variables")
)

summary_network   = bf.networks.TimeSeriesNetwork(summary_dim=40)

inference_network = bf.networks.CouplingFlow()


workflow_RULEXJ = bf.BasicWorkflow(
    simulator         = simulator_RULEXJ,
    adapter           = adapter_RULEXJ,
    summary_network   = summary_network,
    inference_network = inference_network,
)


## 2.4 Mapping Model (MAPP)

In [14]:
# Change cue direction and redefine exemplar and stimuli cues for the mapping model
all_mapp_cues  = mf.preprocess_cues(all_cues,all_crit)
ex_mapp_cues   = all_mapp_cues[ex_IDs-1,:]
mapp_cues      = all_mapp_cues[test_IDs-1,:]
dict_mapp_cues = {f"cue_{i}": mapp_cues[:, i] for i in range(mapp_cues.shape[1])}

In [15]:
def prior_MAPP(lower = 2, upper = 12, rate = rate):

    n_cats = fn.truncated_poisson_np(5, lower = lower, upper = upper)
    sigma  = rng.exponential(1/rate)
    
    return  dict(n_cats=n_cats[0], sigma=sigma)

In [16]:
def model_MAPP(n_cats, sigma, cues=mapp_cues,ex_cues=ex_mapp_cues, ex_crit=ex_crit, p=position_encodings):

    n_trials, _   =  cues.shape 

    # Make predictions based on the GCM model for each person
    pred_crit = mf.MAPP_experiment(n_cats,cues,ex_cues,ex_crit)

    # Simulate responses 
    x = fn.truncnorm_r(mean=pred_crit, sd=sigma, low=0, upp=100, size=n_trials)
  

    return dict(x=x, p=p)

In [17]:
simulator_MAPP = bf.make_simulator([prior_MAPP, model_MAPP])

In [18]:
adapter_MAPP = (
    bf.adapters.Adapter()
    .convert_dtype("float64", "float32")
    .as_time_series(["x","p"])

    .constrain("sigma", lower=0)
    .constrain("n_cats", lower=0)
    
    
    .standardize()

    .concatenate(["n_cats", "sigma"], into="inference_variables")
    .concatenate(["x", "p"], into="summary_variables")
)

summary_network   = bf.networks.TimeSeriesNetwork(summary_dim=15)

inference_network = bf.networks.CouplingFlow()

workflow_MAPP = bf.BasicWorkflow(
    simulator         = simulator_MAPP,
    adapter           = adapter_MAPP,
    summary_network   = summary_network,
    inference_network = inference_network,
)


# 3 Load trained Approximators

In [24]:
workflow_CAM.approximator     = keras.saving.load_model("..\..\Results\Trained Networks\estimation_CAM_FOOD.keras")
workflow_GCM.approximator     = keras.saving.load_model("..\..\Results\Trained Networks\estimation_GCM_FOOD.keras")
workflow_RULEXJ.approximator  = keras.saving.load_model("..\..\Results\Trained Networks\estimation_RULEXJ_FOOD.keras")
workflow_MAPP.approximator    = keras.saving.load_model("..\..\Results\Trained Networks\estimation_MAPP_FOOD.keras")

c:\Users\dizydorc\OneDrive\University\Project Github Repositories\Estimation processes in real-world domains\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 1 variables whereas the saved optimizer has 205 variables. 


In [25]:
df              = pd.read_csv('../../Data/data_analysis_food.csv', sep = ",")

# Replace the NaNs with the row means calculated across all other participants (i.e., exclude the NaN and the participant with the NaN from the mean)

meta_cols     = ['ID_item', 'item', 'img', 'training', 'crit']
response_cols = [col for col in df.columns if col not in meta_cols]

# Copy metadata to preserve original
meta_df   = df[meta_cols].copy()
responses = df[response_cols].copy()

# Function to replace NaNs with mean of other participants
def replace_nan_with_row_mean_except_self(row):
    for col in row.index:
        if pd.isna(row[col]):
            # Compute mean of the row excluding the NaN and this column
            other_values = row.drop(labels=col).dropna()
            if not other_values.empty:
                row[col] = other_values.mean()
    return row

# Apply the function row-wise
responses_filled = responses.apply(replace_nan_with_row_mean_except_self, axis=1)

# Combine with metadata
df_filled = pd.concat([meta_df, responses_filled], axis=1)


data            = df_filled.to_numpy()
data            = np.float32(data[(test_IDs-1),5:])

n_trials, n_sub = data.shape

print("n_trials:", n_trials)
print("n_sub:"   , n_sub)


domain  = "Food"

n_trials: 68
n_sub: 46


# 5 Generate predictions per person and trial

In [26]:
num_samples = 5000
crit        = testing["crit"]

In [27]:
CAM_pred = []

for i in range(n_sub):

    temp           = dict(x = data[:,i].reshape(1,n_trials), p=position_encodings)
    post_samples   = workflow_CAM.sample(num_samples=num_samples, conditions=temp)

    temp_preds = []

    for j in range(num_samples):
        preds = model_CAM(post_samples["w"][0,j,:], np.clip(post_samples["sigma"][:,j,0],0.001,1000))
        temp_preds.append(preds["x"])

    CAM_pred.append(np.c_[np.repeat(i,n_trials),np.repeat("CAM",n_trials),np.repeat(domain,n_trials),range(n_trials),data[:,i],crit,np.array(temp_preds).transpose()])

df_CAM = pd.DataFrame(np.array(CAM_pred).reshape(n_sub*n_trials,5006),columns=["ID_ind","model","domain","test_trial","est","crit"]  + [f"i{i}" for i in range(num_samples)])

In [32]:
GCM_pred = []

for i in range(n_sub):

    temp           = dict(x = data[:,i].reshape(1,n_trials), p=position_encodings)
    post_samples   = workflow_GCM.sample(num_samples=num_samples, conditions=temp)

    temp_preds = []

    for j in range(num_samples):
        preds =  model_GCM(np.clip(post_samples["c"][0,j,0],0.0001,1000), np.clip(post_samples["w"][0,j,:],0.00001,n_dim)/np.sum(np.clip(post_samples["w"][0,j,:],0.00001,n_dim))*n_dim, np.clip(post_samples["sigma"][0,j,0],0.0001,1000))
        temp_preds.append(preds["x"])

    GCM_pred.append(np.c_[np.repeat(i,n_trials),np.repeat("GCM",n_trials),np.repeat(domain,n_trials),range(n_trials),data[:,i],crit,np.array(temp_preds).transpose()])

df_GCM = pd.DataFrame(np.array(GCM_pred).reshape(n_sub*n_trials,5006),columns=["ID_ind","model","domain","test_trial","est","crit"]  + [f"i{i}" for i in range(num_samples)])

In [33]:
RULEXJ_pred = []

for i in range(n_sub):

    temp           = dict(x = data[:,i].reshape(1,n_trials), p=position_encodings)
    post_samples   = workflow_RULEXJ.sample(num_samples=num_samples, conditions=temp)

    temp_preds = []

    for j in range(num_samples):
        preds = model_RULEXJ(post_samples["alpha"][0,j,0],post_samples["w_CAM"][0,j,:],np.clip(post_samples["c"][0,j,0],0.0001,1000), np.clip(post_samples["w_GCM"][0,j,:],0.00001,n_dim)/np.sum(np.clip(post_samples["w_GCM"][0,j,:],0.00001,n_dim))*n_dim, np.clip(post_samples["sigma"][0,j,0],0.0001,1000))
        temp_preds.append(preds["x"])

    RULEXJ_pred.append(np.c_[np.repeat(i,n_trials),np.repeat("RULEXJ",n_trials),np.repeat(domain,n_trials),range(n_trials),data[:,i],crit,np.array(temp_preds).transpose()])

df_RULEXJ = pd.DataFrame(np.array(RULEXJ_pred).reshape(n_sub*n_trials,5006),columns=["ID_ind","model","domain","test_trial","est","crit"]  + [f"i{i}" for i in range(num_samples)])

In [34]:
MAPP_pred = []

for i in range(n_sub):

    temp           = dict(x = data[:,i].reshape(1,n_trials), p=position_encodings)
    post_samples   = workflow_MAPP.sample(num_samples=num_samples, conditions=temp)

    temp_preds = []

    for j in range(num_samples):
        preds = model_MAPP(int(np.round(post_samples["n_cats"][0,j,0],0)), np.clip(post_samples["sigma"][0,j,0],0.0001,1000))
        temp_preds.append(preds["x"])

    MAPP_pred.append(np.c_[np.repeat(i,n_trials),np.repeat("MAPP",n_trials),np.repeat(domain,n_trials),range(n_trials),data[:,i],crit,np.array(temp_preds).transpose()])

df_MAPP = pd.DataFrame(np.array(MAPP_pred).reshape(n_sub*n_trials,5006),columns=["ID_ind","model","domain","test_trial","est","crit"]  + [f"i{i}" for i in range(num_samples)])

In [35]:
df_combined = pd.concat([df_RULEXJ, df_CAM, df_GCM, df_MAPP], axis=0, ignore_index=True)

df_combined.to_csv("../../Results/Parameter Estimates/pred_FOOD_27082025.csv")